In [2]:
import ee
import geemap
import os

ee.Initialize(project="albaniasat")
print("GEE connected!")

GEE connected!


In [3]:
albania = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
            .filter(ee.Filter.eq("country_na", "Albania"))

print("Albania boundary loaded")

Albania boundary loaded


In [5]:
# Clean composite with all 4 bands, summer only, cloud-free
composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B4", "B3", "B2", "B8"])
                .median())

print("Composite image ready!")
info = composite.getInfo()
print("Bands:", [b["id"] for b in info["bands"]])

Composite image ready!
Bands: ['B4', 'B3', 'B2', 'B8']


In [6]:
# Load CORINE Land Cover map
corine = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")

# Clip it to Albania only
corine_albania = corine.clip(albania.geometry())

# Check what classes exist in Albania
values = corine_albania.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=albania.geometry(),
    scale=100,
    maxPixels=1e9
).getInfo()

print("Land cover classes found in Albania:")
for code, count in sorted(values["landcover"].items(), key=lambda x: -x[1]):
    print(f"  Class {code}: {int(count)} pixels")

Land cover classes found in Albania:
  Class 311: 577016 pixels
  Class 324: 394805 pixels
  Class 321: 308616 pixels
  Class 323: 285155 pixels
  Class 243: 248968 pixels
  Class 211: 203390 pixels
  Class 242: 195793 pixels
  Class 333: 143038 pixels
  Class 231: 77403 pixels
  Class 312: 74283 pixels
  Class 112: 65517 pixels
  Class 313: 59311 pixels
  Class 223: 48901 pixels
  Class 512: 44968 pixels
  Class 322: 23377 pixels
  Class 332: 19883 pixels
  Class 331: 19643 pixels
  Class 222: 13859 pixels
  Class 212: 12825 pixels
  Class 511: 8382 pixels
  Class 334: 5845 pixels
  Class 421: 5200 pixels
  Class 521: 4639 pixels
  Class 121: 4247 pixels
  Class 411: 4234 pixels
  Class 523: 3659 pixels
  Class 221: 3599 pixels
  Class 131: 3242 pixels
  Class 335: 2293 pixels
  Class 133: 1159 pixels
  Class 124: 1062 pixels
  Class 422: 947 pixels
  Class 111: 663 pixels
  Class 122: 486 pixels
  Class 142: 353 pixels
  Class 141: 254 pixels
  Class 241: 237 pixels
  Class 123: 159 

## CORINE Land Cover Classes Found in Albania

| Code | Class Name                     | Pixels  |
|------|--------------------------------|---------|
| 311  | Broad-leaved forest            | 577,016 |
| 324  | Transitional woodland/shrub    | 394,805 |
| 321  | Natural grassland              | 308,616 |
| 323  | Sclerophyllous vegetation      | 285,155 |
| 243  | Agriculture + natural veg.     | 248,968 |
| 211  | Non-irrigated arable land      | 203,390 |
| 242  | Complex cultivation patterns   | 195,793 |
| 333  | Sparsely vegetated areas       | 143,038 |
| 231  | Pastures                       | 77,403  |
| 312  | Coniferous forest              | 74,283  |
| 112  | Discontinuous urban fabric     | 65,517  |
| 313  | Mixed forest                   | 59,311  |
| 223  | Olive groves                   | 48,901  |
| 512  | Water bodies                   | 44,968  |

## AlbaniaSAT Class Groupings

| AlbaniaSAT Class | CORINE Codes  |
|------------------|---------------|
| Broad-leaved Forest | 311        |
| Coniferous Forest   | 312        |
| Shrubland           | 323, 324   |
| Agricultural        | 211, 242, 243 |
| Grassland           | 231, 321   |
| Olive Groves        | 223        |
| Urban               | 112        |
| Water               | 512        |

1. Spectral Honesty

Broad-leaved and coniferous forests reflect light differently, broad-leaved trees have wide flat leaves, coniferous have thin needles. Sentinel-2 can clearly see this difference, especially in the NIR band. Merging them into one class would be throwing away real signal.

2. A More Meaningful Benchmark

Finer-grained classes make the dataset more detailed than EuroSAT, not less. A model that can tell broad-leaved from coniferous forest is a more capable model than one that just says "forest." That's a better result to publish.

In [7]:
# Define 8 classes with their CORINE codes
classes = {
    "Broad-leaved Forest": [311],
    "Coniferous Forest": [312],
    "Shrubland": [323, 324],
    "Agricultural": [211, 242, 243],
    "Grassland": [231, 321],
    "Olive Groves": [223],
    "Urban": [112],
    "Water": [512],
}

N_PATCHES = 500
all_samples = {}

for class_name, codes in classes.items():
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))

    points = mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES,
        classBand="landcover",
        region=albania.geometry(),
        scale=100,
        seed=42,
        geometries=True
    )

    count = points.size().getInfo()
    all_samples[class_name] = points
    print(f"{class_name}: {count} sample points found")

Broad-leaved Forest: 500 sample points found
Coniferous Forest: 500 sample points found
Shrubland: 500 sample points found
Agricultural: 500 sample points found
Grassland: 500 sample points found
Olive Groves: 500 sample points found
Urban: 500 sample points found
Water: 500 sample points found


In [24]:
PATCH_SIZE = 32  # 32 pixels each direction = 64x64 total

patches = composite.select(["B4", "B3", "B2", "B8"]).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name, points in all_samples.items():
    clean_name = class_name.replace(" ", "_")
    
    samples = patches.sampleRegions(
        collection=points,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f"patches_{clean_name}",
        folder="AlbaniaSAT_v3",
        fileNamePrefix=f"{clean_name}",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued: {class_name}")

print("All 8 tasks queued!")

Queued: Broad-leaved Forest
Queued: Coniferous Forest
Queued: Shrubland
Queued: Agricultural
Queued: Grassland
Queued: Olive Groves
Queued: Urban
Queued: Water
All 8 tasks queued!


## v2 — Expanded Dataset (1000 patches per class)
Goal: increase from 500 to 1000 patches per class to reduce overfitting.

In [8]:
# Load existing v1 sample points and buffer them
# This ensures new points are at least 640m away from existing ones

N_PATCHES_V2 = 500  # 500 new + 500 existing = 1000 total per class
all_samples_v2 = {}

for class_name, codes in classes.items():
    # Build class mask
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))
    
    # Get existing v1 points for this class
    existing = all_samples[class_name]
    
    # Create 640m exclusion zone around existing points
    exclusion_zone = existing.geometry().buffer(640)
    
    # Convert exclusion zone to a mask
    exclusion_mask = ee.Image.constant(1).clip(albania.geometry()).paint(
        featureCollection=ee.FeatureCollection([ee.Feature(exclusion_zone)]),
        color=0
    )
    
    # Apply exclusion to class mask
    clean_mask = mask.updateMask(exclusion_mask)
    
    # Sample from remaining area
    points = clean_mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES_V2,
        classBand="landcover",
        region=albania.geometry(),
        scale=100,
        seed=123,
        geometries=True
    )
    
    count = points.size().getInfo()
    all_samples_v2[class_name] = points
    print(f"{class_name}: {count} new points found")

Broad-leaved Forest: 500 new points found
Coniferous Forest: 500 new points found
Shrubland: 500 new points found
Agricultural: 500 new points found
Grassland: 500 new points found
Olive Groves: 500 new points found
Urban: 500 new points found
Water: 500 new points found


In [8]:
PATCH_SIZE = 32

patches_v2 = composite.select(["B4", "B3", "B2", "B8"]).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name, points in all_samples_v2.items():
    clean_name = class_name.replace(" ", "_")
    
    samples = patches_v2.sampleRegions(
        collection=points,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f"patches_v2_{clean_name}",
        folder="AlbaniaSAT_v4",
        fileNamePrefix=f"{clean_name}_v2",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued: {class_name}")

print("All 8 tasks queued!")

Queued: Broad-leaved Forest
Queued: Coniferous Forest
Queued: Shrubland
Queued: Agricultural
Queued: Grassland
Queued: Olive Groves
Queued: Urban
Queued: Water
All 8 tasks queued!


In [9]:
import json
import numpy as np
import os

RAW_V1 = "../data/AlbaniaSAT/raw"
RAW_V2 = "../data/AlbaniaSAT/raw_v2"
OUTPUT_PATH = "../data/AlbaniaSAT/processed_v2"
os.makedirs(OUTPUT_PATH, exist_ok=True)

BANDS = ["B4", "B3", "B2", "B8"]

class_names = [
    "Broad-leaved_Forest",
    "Coniferous_Forest",
    "Shrubland",
    "Agricultural",
    "Grassland",
    "Olive_Groves",
    "Urban",
    "Water"
]

all_patches = []
all_labels = []

for label_idx, class_name in enumerate(class_names):
    # Load v1
    with open(os.path.join(RAW_V1, f"{class_name}.geojson")) as f:
        data_v1 = json.load(f)
    
    # Load v2
    with open(os.path.join(RAW_V2, f"{class_name}_v2.geojson")) as f:
        data_v2 = json.load(f)
    
    # Merge features
    all_features = data_v1["features"] + data_v2["features"]
    
    patches = []
    for feature in all_features:
        props = feature["properties"]
        bands = np.array([props[b] for b in BANDS], dtype=np.float32)
        bands = bands[:, :64, :64]
        patches.append(bands)
    
    patches = np.array(patches)
    labels = np.full(len(patches), label_idx, dtype=np.int64)
    
    all_patches.append(patches)
    all_labels.append(labels)
    print(f"{class_name}: {len(patches)} patches")

X = np.concatenate(all_patches, axis=0)
y = np.concatenate(all_labels, axis=0)

print(f"\nFinal dataset: {X.shape} patches, {y.shape} labels")

np.save(os.path.join(OUTPUT_PATH, "patches.npy"), X)
np.save(os.path.join(OUTPUT_PATH, "labels.npy"), y)
print("Saved to processed_v2!")

Broad-leaved_Forest: 1000 patches
Coniferous_Forest: 1000 patches
Shrubland: 1000 patches
Agricultural: 1000 patches
Grassland: 1000 patches
Olive_Groves: 1000 patches
Urban: 1000 patches
Water: 1000 patches

Final dataset: (8000, 4, 64, 64) patches, (8000,) labels
Saved to processed_v2!


## v3 — 6-band export (B2, B3, B4, B8, B11, B12)
Adding SWIR bands to resolve vegetation class confusion.

In [10]:
# New composite with 6 bands including SWIR
composite_6band = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B2", "B3", "B4", "B8", "B11", "B12"])
                .median())

print("6-band composite ready!")
info = composite_6band.getInfo()
print("Bands:", [b["id"] for b in info["bands"]])

6-band composite ready!
Bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']


In [11]:
PATCH_SIZE = 32

patches_6band = composite_6band.select(["B2", "B3", "B4", "B8", "B11", "B12"]).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name in classes.keys():
    clean_name = class_name.replace(" ", "_")
    
    # Merge v1 and v2 sample points
    points_combined = all_samples[class_name].merge(all_samples_v2[class_name])
    
    samples = patches_6band.sampleRegions(
        collection=points_combined,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f"patches_6band_{clean_name}",
        folder="AlbaniaSAT_v5",
        fileNamePrefix=f"{clean_name}_6band",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued: {class_name}")

print("All 8 tasks queued!")

Queued: Broad-leaved Forest
Queued: Coniferous Forest
Queued: Shrubland
Queued: Agricultural
Queued: Grassland
Queued: Olive Groves
Queued: Urban
Queued: Water
All 8 tasks queued!


In [12]:
PATCH_SIZE = 32

patches_6band = composite_6band.neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

# Export v1 and v2 points separately to stay under GEE memory limit
for class_name in classes.keys():
    clean_name = class_name.replace(" ", "_")
    
    # V1 points — 500 patches
    samples_v1 = patches_6band.sampleRegions(
        collection=all_samples[class_name],
        scale=10,
        geometries=True
    )
    task_v1 = ee.batch.Export.table.toDrive(
        collection=samples_v1,
        description=f"6band_v1_{clean_name}",
        folder="AlbaniaSAT_v5",
        fileNamePrefix=f"{clean_name}_6band_v1",
        fileFormat="GeoJSON"
    )
    task_v1.start()
    
    # V2 points — 500 patches
    samples_v2 = patches_6band.sampleRegions(
        collection=all_samples_v2[class_name],
        scale=10,
        geometries=True
    )
    task_v2 = ee.batch.Export.table.toDrive(
        collection=samples_v2,
        description=f"6band_v2_{clean_name}",
        folder="AlbaniaSAT_v5",
        fileNamePrefix=f"{clean_name}_6band_v2",
        fileFormat="GeoJSON"
    )
    task_v2.start()
    
    print(f"Queued v1 + v2: {class_name}")

print("All 16 tasks queued!")

Queued v1 + v2: Broad-leaved Forest
Queued v1 + v2: Coniferous Forest
Queued v1 + v2: Shrubland
Queued v1 + v2: Agricultural
Queued v1 + v2: Grassland
Queued v1 + v2: Olive Groves
Queued v1 + v2: Urban
Queued v1 + v2: Water
All 16 tasks queued!


## v4 — 9-band export (B2, B3, B4, B5, B6, B7, B8, B11, B12)
Adding Red Edge bands to improve forest class separation.

In [13]:
# New composite with 9 bands including Red Edge
composite_9band = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"])
                .median())

print("9-band composite ready!")
info = composite_9band.getInfo()
print("Bands:", [b["id"] for b in info["bands"]])

9-band composite ready!
Bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B11', 'B12']


In [15]:
PATCH_SIZE = 32

patches_9band = composite_9band.select(
    ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]
).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name in classes.keys():
    clean_name = class_name.replace(" ", "_")
    
    # Split v1 and v2 points separately — one task each
    points_v1 = all_samples[class_name]
    points_v2 = all_samples_v2[class_name]
    
    for version, points in [("v1", points_v1), ("v2", points_v2)]:
        samples = patches_9band.sampleRegions(
            collection=points,
            scale=10,
            geometries=True
        )
        
        task = ee.batch.Export.table.toDrive(
            collection=samples,
            description=f"patches_9band_{clean_name}_{version}",
            folder="AlbaniaSAT_v6",
            fileNamePrefix=f"{clean_name}_9band_{version}",
            fileFormat="GeoJSON"
        )
        task.start()
        print(f"Queued: {class_name} {version}")

print("All 16 tasks queued!")

Queued: Broad-leaved Forest v1
Queued: Broad-leaved Forest v2
Queued: Coniferous Forest v1
Queued: Coniferous Forest v2
Queued: Shrubland v1
Queued: Shrubland v2
Queued: Agricultural v1
Queued: Agricultural v2
Queued: Grassland v1
Queued: Grassland v2
Queued: Olive Groves v1
Queued: Olive Groves v2
Queued: Urban v1
Queued: Urban v2
Queued: Water v1
Queued: Water v2
All 16 tasks queued!


## Phase 2 — ESA WorldCover 2021 Labels
Switching from CORINE (100m) to WorldCover (10m) for vegetation classes.
First step: validate WorldCover quality over Albania before full re-export.

In [4]:
# Load WorldCover 2021
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
worldcover_albania = worldcover.clip(albania.geometry())

# Check what classes exist in Albania
wc_values = worldcover_albania.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=albania.geometry(),
    scale=10,
    maxPixels=1e10
).getInfo()

# WorldCover class names
wc_classes = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare/sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen"
}

print("WorldCover classes in Albania:")
for code, count in sorted(wc_values["Map"].items(), key=lambda x: -x[1]):
    name = wc_classes.get(int(code), "Unknown")
    print(f"  Class {code} ({name}): {int(count):,} pixels")

WorldCover classes in Albania:
  Class 10 (Tree cover): 177,507,488 pixels
  Class 30 (Grassland): 133,863,768 pixels
  Class 40 (Cropland): 26,178,544 pixels
  Class 20 (Shrubland): 20,290,033 pixels
  Class 80 (Permanent water): 8,878,200 pixels
  Class 50 (Built-up): 6,982,660 pixels
  Class 60 (Bare/sparse vegetation): 5,297,083 pixels
  Class 90 (Herbaceous wetland): 2,141,808 pixels
  Class 70 (Snow and ice): 112 pixels
  Class 100 (Moss and lichen): 43 pixels


### Finding

WorldCover 2021 provides sufficient pixel coverage for all 5 target classes at 10m resolution. Shrubland has 20M pixels — compared to CORINE's sparse 100m coverage — which is expected to produce significantly cleaner sample points with less boundary noise. Tree cover is not directly usable for forest type separation (WorldCover does not distinguish broad-leaved from coniferous) so CORINE is retained for Broad-leaved Forest, Coniferous Forest, and Olive Groves.

### Label Strategy

| Class | Label Source | Reason |
|---|---|---|
| Broad-leaved Forest | CORINE 311 | WorldCover does not distinguish forest types |
| Coniferous Forest | CORINE 312 | WorldCover does not distinguish forest types |
| Olive Groves | CORINE 223 | WorldCover does not distinguish olive groves |
| Shrubland | WorldCover 20 | 10m labels, cleaner boundaries than CORINE |
| Grassland | WorldCover 30 | 10m labels, cleaner boundaries than CORINE |
| Agricultural | WorldCover 40 | 10m labels, cleaner boundaries than CORINE |
| Urban | WorldCover 50 | 10m labels, cleaner boundaries than CORINE |
| Water | WorldCover 80 | 10m labels, cleaner boundaries than CORINE |

In [6]:
# Load CORINE
corine = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")
corine_albania = corine.clip(albania.geometry())

# Load WorldCover
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
worldcover_albania = worldcover.clip(albania.geometry())

# 6-band composite
composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B2", "B3", "B4", "B8", "B11", "B12"])
                .median())

# WorldCover classes
classes_wc = {
    "Shrubland":    [20],
    "Grassland":    [30],
    "Agricultural": [40],
    "Urban":        [50],
    "Water":        [80],
}

# CORINE classes
classes_corine = {
    "Broad-leaved_Forest": [311],
    "Coniferous_Forest":   [312],
    "Olive_Groves":        [223],
}

N_PATCHES = 1000
all_samples_p2 = {}

# Sample WorldCover classes
for class_name, codes in classes_wc.items():
    mask = worldcover_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(worldcover_albania.eq(code))
    
    points = mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES,
        classBand="Map",
        region=albania.geometry(),
        scale=10,
        seed=42,
        geometries=True
    )
    
    count = points.size().getInfo()
    all_samples_p2[class_name] = points
    print(f"{class_name} (WorldCover): {count} points")

# Sample CORINE classes
for class_name, codes in classes_corine.items():
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))
    
    points = mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES,
        classBand="landcover",
        region=albania.geometry(),
        scale=100,
        seed=42,
        geometries=True
    )
    
    count = points.size().getInfo()
    all_samples_p2[class_name] = points
    print(f"{class_name} (CORINE): {count} points")

Shrubland (WorldCover): 1000 points
Grassland (WorldCover): 1000 points
Agricultural (WorldCover): 1000 points
Urban (WorldCover): 1000 points
Water (WorldCover): 1000 points
Broad-leaved_Forest (CORINE): 1000 points
Coniferous_Forest (CORINE): 1000 points
Olive_Groves (CORINE): 1000 points


In [7]:
PATCH_SIZE = 32

patches_p2 = composite.select(["B2", "B3", "B4", "B8", "B11", "B12"]).neighborhoodToArray(
    kernel=ee.Kernel.rectangle(PATCH_SIZE, PATCH_SIZE, "pixels")
)

for class_name, points in all_samples_p2.items():
    clean_name = class_name.replace(" ", "_")
    
    samples = patches_p2.sampleRegions(
        collection=points,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f"p2_{clean_name}",
        folder="AlbaniaSAT_Phase2",
        fileNamePrefix=f"{clean_name}_p2",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued: {class_name}")

print("All 8 tasks queued!")

Queued: Shrubland
Queued: Grassland
Queued: Agricultural
Queued: Urban
Queued: Water
Queued: Broad-leaved_Forest
Queued: Coniferous_Forest
Queued: Olive_Groves
All 8 tasks queued!


### Phase 2 — Sampling (WorldCover + CORINE Hybrid)

1,000 points per class sampled successfully across all 8 classes — WorldCover 2021 (10m) for Shrubland, Grassland, Agricultural, Urban, Water and CORINE 2018 for Broad-leaved Forest, Coniferous Forest, Olive Groves. 8 export tasks queued to AlbaniaSAT_Phase2 on Google Drive.

Note: purity filter (200m erosion) was skipped — caused GEE timeout at 10m resolution. WorldCover's native 10m resolution partially compensates.